# Random Head Knockout Baseline — Instruct Models

Reproduces **Table 11** of the paper: for each instruct model, 100 random
draws of attention heads sampled from the same layers (and in the same
number per layer) as the identified cultural-binding heads, each knocked
out on the B→item edge. The empirical p-value compares the identified
heads' Δ(S) reduction against this null distribution; a cluster-corrected
paired t-test is also reported for the identified-heads knockout. Outputs:
a histogram figure and a results pickle under `./results/<model>_instruct/`.

Rebuilt from `pipeline_instruct.ipynb` (canonical master instruct
pipeline), cell 20 ("Random head knockout baseline — standalone"), with
imports and model loading taken from cells 2 and 10.

Run once per `MODEL_KEY`. Random-trial seeds are `1000 + trial`, preserved
verbatim from the source.


In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma2", "nemo"}


In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
DATA_DIR = config.DATA_DIR
HF_TOKEN = config.HF_TOKEN
OUTPUT_DIR = config.OUTPUT_DIR
ACTIVE_MODEL = config.ACTIVE_MODEL  # historical alias used by the source cell

import torch
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

from common.text_parsers import extract_options
from common.instruct.data import load_n4, build_factorial_as_conditions
from common.instruct.prompts import format_for_chat, find_option_token_ids, detect_spans
from common.instruct.hooks import compute_logit_scores_edge


In [ ]:
# ── Load model (verbatim from pipeline_instruct.ipynb, Stage 1 cell) ──
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

first_device = next(model.parameters()).device

# Register runtime singletons so common helpers can see them
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device


In [ ]:
# ================================================================
# RANDOM HEAD KNOCKOUT BASELINE — STANDALONE
# Requires: imports + config + shared helpers + edge knockout, model loaded.
# ================================================================
import random
import pickle
import numpy as np
from scipy.stats import ttest_rel

N_RANDOM_TRIALS = 100
RANDOM_SEED_START = 1000
SAVE_DIR = Path(OUTPUT_DIR)
SAVE_DIR.mkdir(exist_ok=True)

FINAL_HEADS = CFG['heads']
n_attn_heads = model.config.num_attention_heads

print("=" * 80)
print(f"RANDOM HEAD KNOCKOUT BASELINE — {CFG['label']}")
print(f"  Identified heads: {FINAL_HEADS}")
print(f"  Heads per layer:  {n_attn_heads}")
print(f"  Random trials:    {N_RANDOM_TRIALS}")
print("=" * 80)

# ── 1. Load data ──
print("\n  Loading data...")
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
conditions = ['B_cult', 'B_unrel']
print(f"  {n_total} factorial pairs")

# ── 2. Option tokens ──
option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

# ── 3. Format texts + build positions ──
print("  Building positions...")
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}
positions = {c: [] for c in conditions}
n_valid = 0

for c in conditions:
    for i in range(n_total):
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer,
                          item_required=True)
        if sp is None:
            positions[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions[c].append({
                'item_tokens': sp['item'], 'B_tokens': B_tokens,
                'A_tokens': A_tokens,
            })
            if c == conditions[0]:
                n_valid += 1
        del enc

print(f"  Valid positions: {n_valid}/{n_total}")

# ── 4. Compute baseline (no knockout) ──
print("\n  Computing baseline...")
scores_base = {}
for cond in conditions:
    scores_base[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        {}, positions[cond], 'B_to_item', option_tokens)
diffs_base = scores_base['B_cult'] - scores_base['B_unrel']
delta_base_val = diffs_base.mean()
print(f"  Baseline Δ(S) = {delta_base_val:.4f}")

# ── 5. Compute real heads knockout ──
print("  Computing real heads knockout...")
scores_real = {}
for cond in conditions:
    scores_real[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        FINAL_HEADS, positions[cond], 'B_to_item', option_tokens)
diffs_real = scores_real['B_cult'] - scores_real['B_unrel']
real_delta = diffs_real.mean()
real_reduction = (1 - real_delta / delta_base_val) * 100
print(f"  Real heads Δ(S) = {real_delta:.4f}  (reduction: {real_reduction:.1f}%)")

# ── 6. Build random pool ──
identified_flat = set()
heads_per_layer = {}
for layer, head_list in FINAL_HEADS.items():
    heads_per_layer[layer] = len(head_list)
    for h in head_list:
        identified_flat.add((layer, h))

candidate_pool = {}
for layer, n_to_sample in heads_per_layer.items():
    pool = [h for h in range(n_attn_heads)
            if (layer, h) not in identified_flat]
    candidate_pool[layer] = pool
    print(f"  Layer {layer}: {n_to_sample} to sample from "
          f"{len(pool)} candidates")

# ── 7. Run random trials ──
print(f"\n  Running {N_RANDOM_TRIALS} random trials...")
random_reductions = []

for trial in range(N_RANDOM_TRIALS):
    rng = random.Random(RANDOM_SEED_START + trial)

    # Sample random heads from the same layers
    rand_heads = {}
    for layer, n_to_sample in heads_per_layer.items():
        sampled = rng.sample(candidate_pool[layer], n_to_sample)
        rand_heads[layer] = sampled

    # B→item knockout with random heads
    scores_rand = {}
    for cond in conditions:
        scores_rand[cond] = compute_logit_scores_edge(
            model, tokenizer, texts_fmt[cond], data[cond],
            rand_heads, positions[cond], 'B_to_item', option_tokens)

    delta_rand = (scores_rand['B_cult'] - scores_rand['B_unrel']).mean()
    red_rand = (1 - delta_rand / delta_base_val) * 100
    random_reductions.append(red_rand)

    if (trial + 1) % 10 == 0:
        print(f"    Trial {trial+1:3d}/{N_RANDOM_TRIALS}  "
              f"reduction: {red_rand:+.1f}%  "
              f"(running median: {np.median(random_reductions):.1f}%)")

random_reductions = np.array(random_reductions)

# ── 8. Results ──
print(f"\n{'='*80}")
print(f"RESULTS — {CFG['label']}")
print(f"{'='*80}")
print(f"  Identified heads:  {real_reduction:.1f}% reduction")
print(f"  Random (n={N_RANDOM_TRIALS}):")
print(f"    Mean ± Std:      {random_reductions.mean():.1f}% "
      f"± {random_reductions.std():.1f}%")
print(f"    Median:          {np.median(random_reductions):.1f}%")
print(f"    [Min, Max]:      [{random_reductions.min():.1f}%, "
      f"{random_reductions.max():.1f}%]")
print(f"    95th pctl:       {np.percentile(random_reductions, 95):.1f}%")
print(f"    99th pctl:       {np.percentile(random_reductions, 99):.1f}%")

# Empirical p-value (one-sided: how often random >= real)
n_exceed = (random_reductions >= real_reduction).sum()
p_empirical = n_exceed / N_RANDOM_TRIALS
print(f"\n  Empirical p-value: {p_empirical:.4f} "
      f"({n_exceed}/{N_RANDOM_TRIALS} random >= real)")

# Cluster-corrected t-test on real heads
items = np.array(data['items_cult'])
unique_items = np.unique(items)
mean_base_item = np.array([diffs_base[items == it].mean()
                           for it in unique_items])
mean_real_item = np.array([diffs_real[items == it].mean()
                           for it in unique_items])
t_clust, p_clust = ttest_rel(mean_base_item, mean_real_item)
print(f"\n  Cluster-corrected t-test (real heads, n={len(unique_items)}):")
print(f"    t = {t_clust:.3f}, p = {p_clust:.2e}")

# ── 9. Plot ──
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(random_reductions, bins=25, alpha=0.7, color='steelblue',
        edgecolor='white', label=f'Random heads (n={N_RANDOM_TRIALS})')
ax.axvline(x=real_reduction, color='red', linewidth=2.5, linestyle='-',
           label=f'Identified heads: {real_reduction:.1f}%')
ax.axvline(x=np.percentile(random_reductions, 95), color='orange',
           linewidth=1.5, linestyle='--', label='95th pctl')
ax.set_xlabel('% reduction in |Δ(S)|', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'{CFG["label"]}: Random vs Identified Head Knockout',
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / f'{ACTIVE_MODEL}_random_ko.png', dpi=150)
plt.show()

# ── 10. Save results ──
results = {
    'model': ACTIVE_MODEL,
    'label': CFG['label'],
    'identified_heads': FINAL_HEADS,
    'n_attn_heads': n_attn_heads,
    'n_trials': N_RANDOM_TRIALS,
    'n_pairs': n_total,
    'n_valid': n_valid,
    'delta_baseline': float(delta_base_val),
    'real_reduction_pct': float(real_reduction),
    'real_delta': float(real_delta),
    'random_reductions_pct': random_reductions.tolist(),
    'random_mean': float(random_reductions.mean()),
    'random_median': float(np.median(random_reductions)),
    'random_std': float(random_reductions.std()),
    'random_p95': float(np.percentile(random_reductions, 95)),
    'random_p99': float(np.percentile(random_reductions, 99)),
    'empirical_p': float(p_empirical),
    'n_exceed': int(n_exceed),
    't_clustered': float(t_clust),
    'p_clustered': float(p_clust),
    'diffs_base': diffs_base.tolist(),
    'diffs_real_ko': diffs_real.tolist(),
    'items': data['items_cult'],
}

pkl_path = SAVE_DIR / f'results_{ACTIVE_MODEL}_instruct_random_ko.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Saved to {pkl_path}")
print("  Done.")
